# 01 - Data Cleaning, OCR Risalah & Pembagian Dataset 80:20

Pipeline:
1. OCR seluruh PDF risalah → simpan teks ke `dataset/02_extracted/ocr_risalah/`
2. Pasangkan transkripsi Whisper (source) dengan teks OCR yang relevan lalu buat ringkasan manual (target)
3. Bersihkan & validasi pasangan data
4. Split otomatis 80% latih / 20% uji → `train.csv` & `test.csv`

## 1. Import Library

In [2]:
import pandas as pd
import re
from pathlib import Path
import sys
sys.path.append('..')
from modules.ocr_risalah import proses_semua_pdf

## 2. OCR Semua Risalah PDF

In [2]:
DIR_PDF    = Path('../dataset/01_raw/risalah_pdf')
DIR_OCR    = Path('../dataset/02_extracted/ocr_risalah')

# Sesuaikan pola_awal dan pola_akhir dengan format risalah kamu!
POLA_AWAL  = r'MENYANYIKAN LAGU INDONESIA RAYA'
POLA_AKHIR = r'RAPAT DITUTUP PUKUL'

hasil_ocr = proses_semua_pdf(
    direktori_pdf=DIR_PDF,
    direktori_output=DIR_OCR,
    pola_awal=POLA_AWAL,
    pola_akhir=POLA_AKHIR,
)
print(f'Total risalah berhasil di-OCR: {len(hasil_ocr)}')

✅ Paripurna_Ke_10_Persidangan_II_2024_2025.pdf → 4094 kata diekstrak
✅ Paripurna_Ke_11_Persidangan_II_2024_2025.pdf → 711 kata diekstrak
✅ Paripurna_Ke_11_Persidangan_III_2023_2024.pdf → 2469 kata diekstrak
✅ Paripurna_Ke_12_Persidangan_III_2023_2024.pdf → 3676 kata diekstrak
✅ Paripurna_Ke_13_Persidangan_IV_2023_2024.pdf → 6099 kata diekstrak
✅ Paripurna_Ke_14_Persidangan_II_2024_2025.pdf → 1613 kata diekstrak
✅ Paripurna_Ke_14_Persidangan_IV_2023_2024.pdf → 9899 kata diekstrak
✅ Paripurna_Ke_15_Persidangan_II_2024_2025.pdf → 6006 kata diekstrak
✅ Paripurna_Ke_15_Persidangan_IV_2023_2024.pdf → 2325 kata diekstrak
✅ Paripurna_Ke_16_Persidangan_II_2024_2025.pdf → 2543 kata diekstrak
✅ Paripurna_Ke_16_Persidangan_V_2023_2024.pdf → 3956 kata diekstrak
✅ Paripurna_Ke_17_Persidangan_III_2024_2025.pdf → 852 kata diekstrak
✅ Paripurna_Ke_17_Persidangan_V_2023_2024.pdf → 5023 kata diekstrak
✅ Paripurna_Ke_18_Persidangan_III_2024_2025.pdf → 5208 kata diekstrak
✅ Paripurna_Ke_18_Persidangan_V_20

## 3. Load Transkripsi Whisper & Pasangkan dengan OCR

In [4]:
DIR_TRANSKRIP = Path('../dataset/02_extracted/whisper_transcripts')

WINDOW_SOURCE = 400
WINDOW_TARGET = 200
STRIDE = 300  # overlap 100 kata

baris = []

for txt_file in sorted(DIR_TRANSKRIP.glob('*.txt')):
    nama_tanpa_ext = txt_file.stem
    ocr_file = DIR_OCR / (nama_tanpa_ext + '.txt')
    
    if not ocr_file.exists():
        print(f'[SKIP] Tidak ada pasangan OCR untuk: {txt_file.name}')
        continue
        
    source = txt_file.read_text(encoding='utf-8').strip()
    target = ocr_file.read_text(encoding='utf-8').strip()
    
    if not source or not target:
        continue

    source_words = source.split()
    target_words = target.split()

    min_length = min(len(source_words), len(target_words))

    # Kalau dokumen terlalu pendek, skip
    if min_length < WINDOW_SOURCE:
        continue

    # Sliding window
    for start in range(0, min_length - WINDOW_SOURCE + 1, STRIDE):
        end_source = start + WINDOW_SOURCE
        end_target = start + WINDOW_TARGET

        if end_target > len(target_words):
            break

        s_chunk = " ".join(source_words[start:end_source])
        t_chunk = " ".join(target_words[start:end_target])

        baris.append({
            "source": s_chunk,
            "target": t_chunk
        })

df = pd.DataFrame(baris)

print(f"\nTotal pasangan data hasil segmentasi: {len(df)} baris")
df.head()


Total pasangan data hasil segmentasi: 519 baris


,source,target
0,Hadirin sekalian. Rami persilakan untuk duduk ...,"Hadirin sekalian, kami persilakan untuk duduk ..."
1,pembukaan masa persidangan 2 Tahun Sidang 2024...,"Sidang 2024-2025. Untuk itu, izinkan saya memb..."
2,mitra kerja pemerintah dan DPR-Ri akan memasti...,Indonesia yang besar ini tentulah membutuhkan ...
3,dilaksanakan sesuai dengan kemampuan keuangan ...,kualitas hidup rakyat. Pendapatan rakyat menin...
4,pangan serta tancar pengetian import beberapa ...,pada lembaga yang membidangi ketertiban dan ke...


In [1]:
import pandas as pd
import re
from pathlib import Path

DIR_TRANSKRIP = Path('../dataset/02_extracted/whisper_transcripts')
DIR_OCR = Path('../dataset/02_extracted/ocr_risalah')
MAX_WORDS = 350 

baris_data = []

print("Sedang memotong Whisper dan memvalidasi keberadaan OCR...")

for txt_file in sorted(DIR_TRANSKRIP.glob('*.txt')):
    source_whisper = txt_file.read_text(encoding='utf-8').strip()
    
    # Validasi file OCR pasangannya
    ocr_file = DIR_OCR / (txt_file.stem + '.txt')
    
    if not source_whisper:
        continue
        
    if not ocr_file.exists():
        print(f"OCR tidak ditemukan untuk {txt_file.stem}, dilewati.")
        continue

    kalimat_list = re.split(r'(?<=[.!?]) +', source_whisper)
    chunk_saat_ini = []
    jumlah_kata_saat_ini = 0

    for kalimat in kalimat_list:
        jml_kata = len(kalimat.split())

        if jumlah_kata_saat_ini + jml_kata > MAX_WORDS and jumlah_kata_saat_ini > 0:
            baris_data.append({
                "dokumen_asal": txt_file.stem,
                "input_whisper_segment": " ".join(chunk_saat_ini),
                "target_summary_manual": ""
            })
            chunk_saat_ini = [kalimat]
            jumlah_kata_saat_ini = jml_kata
        else:
            chunk_saat_ini.append(kalimat)
            jumlah_kata_saat_ini += jml_kata

    if chunk_saat_ini and jumlah_kata_saat_ini > 50:
        baris_data.append({
            "dokumen_asal": txt_file.stem,
            "input_whisper_segment": " ".join(chunk_saat_ini),
            "target_summary_manual": ""
        })

df = pd.DataFrame(baris_data)
df.to_csv('../dataset/data_segment_siap_anotasi.csv', index=False)

print(f"Selesai! {len(df)} segmen siap dianotasi.")

Sedang memotong Whisper dan memvalidasi keberadaan OCR...
Selesai! 486 segmen siap dianotasi.


## 4. Validasi & Bersihkan

In [ ]:
# Load ulang dataset hasil anotasi
df = pd.read_csv('../dataset/data_segment_siap_anotasi.csv')

# Hapus baris yang summary-nya masih kosong
df = df.dropna(subset=['input_whisper_segment', 'target_summary_manual']).reset_index(drop=True)

# Hapus yang summary kosong string
df = df[df['target_summary_manual'].str.strip() != ""]

# Filter minimal panjang input (misal ≥100 kata)
df = df[df['input_whisper_segment'].str.split().str.len() >= 100].reset_index(drop=True)

# Filter panjang summary realistis (misal 30–80 kata)
df = df[
    (df['target_summary_manual'].str.split().str.len() >= 30) &
    (df['target_summary_manual'].str.split().str.len() <= 80)
].reset_index(drop=True)

print(f'Data valid siap latih setelah cleaning: {len(df)} baris')

# Statistik panjang kata
print("\nStatistik Panjang Kata:")
print(
    df[['input_whisper_segment', 'target_summary_manual']]
    .apply(lambda c: c.str.split().str.len())
    .describe()
)

Data valid siap latih setelah cleaning: 30 baris

Statistik Panjang Kata:
       source  target
count    30.0    30.0
mean    400.0   200.0
std       0.0     0.0
min     400.0   200.0
25%     400.0   200.0
50%     400.0   200.0
75%     400.0   200.0
max     400.0   200.0

Data latih (train): 24 (80%)
Data uji  (test) : 6  (20%)


## 5. Split 80:20 → train.csv & test.csv

> Tidak ada validation set. Data dibagi langsung menjadi **data latih (80%)** dan **data uji (20%)**.

In [5]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    shuffle=True,
)
df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f'Data latih (train): {len(df_train)} ({len(df_train)/len(df)*100:.0f}%)')
print(f'Data uji  (test) : {len(df_test)}  ({len(df_test)/len(df)*100:.0f}%)')

Data latih (train): 24 (80%)
Data uji  (test) : 6  (20%)


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset hasil anotasi
df = pd.read_csv('../dataset/data_segment_siap_anotasi.csv')

# Hapus baris yang summary kosong atau NaN
df = df.dropna(subset=['input_whisper_segment', 'target_summary_manual'])
df = df[df['target_summary_manual'].str.strip() != ""].reset_index(drop=True)

print(f'Data siap latih setelah cleaning: {len(df)} baris')

# Split 80:20
df_train, df_test = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    shuffle=True,
)

df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f'\nData latih (train): {len(df_train)} ({len(df_train)/len(df)*100:.0f}%)')
print(f'Data uji  (test) : {len(df_test)}  ({len(df_test)/len(df)*100:.0f}%)')

Data siap latih setelah cleaning: 486 baris

Data latih (train): 388 (80%)
Data uji  (test) : 98  (20%)


## 6. Simpan ke CSV

In [3]:
from pathlib import Path

DATA_DIR = Path('../dataset/03_paired')
DATA_DIR.mkdir(parents=True, exist_ok=True)

df_train.to_csv(DATA_DIR / 'train.csv', index=False, encoding='utf-8')
df_test.to_csv(DATA_DIR  / 'test.csv',  index=False, encoding='utf-8')

print('Dataset berhasil disimpan:')
print(f'  train.csv → {len(df_train)} baris')
print(f'  test.csv  → {len(df_test)} baris')

Dataset berhasil disimpan:
  train.csv → 388 baris
  test.csv  → 98 baris
